# Residual RL on a frozen SmolVLA — cube to box

Trains a small **bounded residual** on top of the frozen SmolVLA policy from `ego2libero`:

```
a = clip(a_base + alpha * a_res, -1, 1)      alpha = 0.2
```

At robosuite's OSC limits that is at most **1 cm** and **5.7°** of correction per step, so the base
stays in charge. On checkpoint `026000` the measured baseline is **45 % in-box (36/80, CI 34–56 %)**,
and **16 of 21 failures never get the cube off the table**, missing the grasp by 1–3 cm — inside the
residual's envelope.

**Why this is worth running on a GPU.** Measured per environment step on a 16-core CPU box:

| component | cost | notes |
|---|---|---|
| frozen SmolVLA forward | **~20 ms** | 1.0–1.4 s per 50-step chunk, saturates every core |
| MuJoCo physics | 4.3 ms | |
| 3-camera render | 0.1 ms | only needed on a chunk boundary |

The base policy is **82 % of the cost** and it is the part a GPU fixes — expect roughly 5–7×.
Past that the bottleneck becomes MuJoCo physics on the runtime's vCPUs, so **Runtime → Change runtime
type → GPU** matters far more than anything else here.

> **Set the runtime to GPU before running.** On CPU this notebook works but is ~6× slower.

## 1. Environment

Everything runs in one interpreter, but **numpy must be pinned below 2**. lerobot pulls
`rerun-sdk`, which asks for `numpy>=2`; rerun is never imported on the inference path, and numpy 2
silently changes what MuJoCo renders — same mujoco, same PyOpenGL, byte-identical physics, images
~14 levels darker. For a vision policy that is off-distribution: the same checkpoint scored 20/40
under numpy 1.26 and 7–9/40 under numpy 2 on identical start poses.

> **Set the runtime to GPU first** — Runtime → Change runtime type → T4 GPU. The frozen base policy
> is ~82 % of per-step cost and is the only part a GPU fixes.

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

# numpy MUST stay < 2. lerobot's rerun-sdk dep asks for >= 2, but rerun is never imported on the
# inference path and lerobot runs fine on 1.26. numpy 2 silently changes what MuJoCo RENDERS --
# same mujoco, same PyOpenGL, byte-identical physics, but images ~14 levels darker. SmolVLA is a
# vision policy, so that is off-distribution: the same checkpoint scored 20/40 under numpy 1.26
# and 7-9/40 under numpy 2 on identical start poses.
pip("lerobot[smolvla]==0.4.4")
pip("robosuite==1.4.1", "mujoco==2.3.7")
pip("numpy==1.26.4")          # last, so nothing re-upgrades it

import numpy, torch
assert numpy.__version__.startswith("1.26"), f"numpy must be 1.26, got {numpy.__version__}"
print("numpy", numpy.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
import os, subprocess, sys

# MuJoCo needs a GL backend for offscreen rendering. EGL requires a GPU; on a CPU-only runtime it
# aborts the process at the C level with no Python traceback, which looks like the cell simply
# ending. Pick the backend from what the runtime actually has.
import torch
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    os.environ["MUJOCO_GL"] = "egl"
else:
    print("No CUDA runtime detected -> Runtime > Change runtime type > GPU is strongly recommended;\n"
          "the frozen base policy is ~82% of per-step cost and only a GPU fixes it.\n"
          "Falling back to software rendering (osmesa) so this at least runs.")
    subprocess.run("apt-get install -y -qq libosmesa6-dev", shell=True, check=False)
    os.environ["MUJOCO_GL"] = "osmesa"
os.environ["OMP_NUM_THREADS"] = "2"   # MuJoCo is the CPU bottleneck once the GPU serves the policy

import numpy as np, robosuite, mujoco
print("numpy", np.__version__, "| robosuite", robosuite.__version__, "| mujoco", mujoco.__version__)
print("MUJOCO_GL =", os.environ["MUJOCO_GL"], "| cuda:", HAS_GPU,
      "|", torch.cuda.get_device_name(0) if HAS_GPU else "CPU only")
assert np.__version__.startswith("2"), "expected numpy 2 in this single-env setup"

## 2. Code and data

Mount Drive and point `DRIVE` at the staged bundle. Its layout is exactly:

```
ego2libero_colab/              926 MB total, 151 files
  checkpoint/
    pretrained_model/          907 MB  inference weights ONLY
  processed/                    20 MB  selection.csv + 64 demo_C*_20hz.{npz,json}
  code/
    ego2libero/                        the 12 modules the trainer imports
    scripts/train_residual.py
    configs/libero_frame.yaml          load_T_L_W reads this
  index.json                           the T_L_W and box the base policy trained in
```

**The scene must match the base policy's training frame.** Checkpoint `026000` was trained under the
pilots' `T_L_W` with the box at `(-0.022, 0.118)`. Evaluated in the current rotated frame it sees a
workspace turned 180° from its training data and scores ~0 for reasons that have nothing to do with
the residual. `index.json` is what pins this down.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys, shutil

DRIVE = Path("/content/drive/MyDrive/ego2libero_colab")
assert DRIVE.exists(), f"upload the bundle to {DRIVE} (layout is listed above)"

SESSION = "2026-10-07_dataset"
WORK = Path("/content/work"); WORK.mkdir(exist_ok=True)
REFRESH = True   # set False to keep a local copy across runs; True always re-copies from Drive

# Code and the small data files come local (fast); the 907 MB checkpoint is read from Drive.
# Always re-copying matters: if you fix something on Drive, a cached /content/work silently
# keeps serving the old files, and the failure looks like a missing module rather than a stale copy.
if REFRESH or not (WORK / "code").exists():
    shutil.rmtree(WORK / "code", ignore_errors=True)
    shutil.copytree(DRIVE / "code", WORK / "code")
proc = WORK / "data/processed" / SESSION
if REFRESH or not proc.exists():
    shutil.rmtree(proc, ignore_errors=True)
    proc.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE / "processed", proc)

CKPT = DRIVE / "checkpoint"
import torch
DEV = "cuda" if torch.cuda.is_available() else "cpu"

# Drop any half-imported copy this kernel is holding, then put the fresh one first on sys.path.
for m in [m for m in sys.modules if m == "ego2libero" or m.startswith("ego2libero.")]:
    del sys.modules[m]
while str(WORK / "code") in sys.path:
    sys.path.remove(str(WORK / "code"))
sys.path.insert(0, str(WORK / "code"))

# Fail loudly and specifically rather than deep inside an import three cells later.
missing = [f for f in ("ego2libero/__init__.py", "ego2libero/sim/__init__.py",
                       "ego2libero/sim/base_policy.py", "ego2libero/sim/residual.py",
                       "ego2libero/sim/scene.py", "ego2libero/sim/frame.py",
                       "ego2libero/rl/sac.py", "ego2libero/export/write_demo.py",
                       "configs/libero_frame.yaml", "scripts/train_residual.py")
           if not (WORK / "code" / f).exists()]
assert not missing, ("these did not make it to Drive: " + ", ".join(missing) +
                     "\nGoogle Drive's uploader skips 0-byte files; re-upload code/ from the bundle.")

import ego2libero
print("code :", list(ego2libero.__path__))
print("data :", proc, f"({len(list(proc.glob('*_20hz.npz')))} demo exports)")
print("ckpt :", CKPT)

## 3. Smoke test — does the baseline reproduce?

Before training anything, confirm the stack runs and the **zero residual** reproduces the base
policy's behaviour. This is the check that validates the observation layout, the action composition
and the render-on-chunk-boundary optimisation all at once.

Note the base policy is **stochastic** — flow matching resamples noise each call, giving per-step
action std of 0.03–0.08. A handful of rollouts will scatter widely around 45 %; that is expected.

In [ ]:
import json, time
import numpy as np
from ego2libero.sim.base_policy import InProcessBasePolicy
from ego2libero.sim.residual import ResidualCubeToBox, ResidualConfig
from ego2libero.sim.frame import load_T_L_W
from ego2libero.export.write_demo import load_demo
import csv

idx = json.loads((DRIVE / "index.json").read_text())
box_xy = np.array(idx["box_xy_L"], float)
T_L_W = np.array(idx["T_L_W"], float) if "T_L_W" in idx else load_T_L_W()   # the frame the base trained in

proc = WORK / "data/processed" / SESSION
rows = [r for r in csv.DictReader(open(proc / "selection.csv")) if r["position"]]
starts, labels = [], []
for p in sorted({r["position"] for r in rows}):
    g = sorted((r for r in rows if r["position"] == p), key=lambda r: -float(r["score"]))
    for r in g[:2]:
        starts.append(T_L_W @ load_demo(proc / f"{r['clip']}_20hz")["T_W_O"][0]); labels.append(p)

base = InProcessBasePolicy(CKPT, device=DEV)
env = ResidualCubeToBox(base, starts, box_xy, ResidualConfig(alpha=0.2),
                        task="pick up the red cube and place it in the box")

t0, n, ok = time.time(), 0, 0
for i in range(len(starts)):
    o = env.reset(i); done = False
    while not done:
        o, r, done, info = env.step(np.zeros(7)); n += 1
    ok += info["success"]
    print(f"  {labels[i]} start {i:2d}: {'SUCCESS' if info['success'] else 'failed '} {info['steps']:3d} steps")
dt = time.time() - t0
print(f"\nzero residual: {ok}/{len(starts)}   |   {n/dt:.0f} env-steps/s on {DEV}")
print("CPU reference was ~45 steps/s; a GPU runtime should be several times that.")

## 4. Train

SAC, because environment steps are expensive — the full argument is in `ego2libero/rl/sac.py`.
In one line: the frozen base costs ~1 s of forward per 50 steps, which puts us firmly in the
low-sample regime, and only an off-policy method can afford it.

Two residual-specific details worth knowing while reading the logs:

- The actor's last layer starts near zero, so the **evaluated** (deterministic) residual begins as
  a no-op and the first `EVAL` line should land near the 45 % baseline. Training episodes use
  *sampled* residuals spanning the full ±0.2 range from step one, so the `success` column in the
  per-episode log starts **below** 45 % and climbs. Judge progress from the `EVAL` lines, not the
  per-episode log.
- `lifted` is the number to watch early. 76 % of baseline failures never get the cube off the table,
  so the residual should move `lifted` before it moves `success`.

Checkpoints go to Drive every `--save-every` steps, so a disconnect costs minutes. Re-run the cell
with `--resume` pointing at the same directory to continue.

In [ ]:
RUN = DRIVE / "runs/residual_026000"
RUN.mkdir(parents=True, exist_ok=True)

import torch
DEV = "cuda" if torch.cuda.is_available() else "cpu"   # defined here too, so this cell stands alone

# Free the GPU only if the smoke-test cell ran in THIS kernel. After a runtime restart those names
# are gone, and the cell should not die on that.
for _name in ("env", "base"):
    try:
        globals()[_name].close()
    except Exception:
        pass

# --data-root tells the trainer where data/processed/<session>/ lives. The code is not sitting in
# the repo here, so without it the script looks under its own parent directory and finds nothing.
!cd {WORK} && python {WORK}/code/scripts/train_residual.py {SESSION} {CKPT} \
    --as-trained {DRIVE}/index.json --data-root {WORK} \
    --in-process --device {DEV} \
    --steps 100000 --alpha 0.2 --per-position 8 \
    --start-steps 2000 --eval-every 10000 --save-every 5000 \
    --out {RUN}

# if the session dropped, re-run this cell with  --resume {RUN}  appended

## 5. Results

`evaluate()` is **paired**: each start pose is run twice under the *same* base-policy noise seed,
once with the residual and once with a zero residual. Because the base's own action noise (std
0.03–0.08) is comparable to the residual's 0.2 authority, an unpaired comparison needs roughly 4×
the rollouts to resolve the same effect.

Read `residual_only_wins` vs `baseline_only_wins`: that is the per-start difference with the base's
noise held fixed, and it is the honest measure of whether the residual helped.

In [ ]:
import json, glob
import numpy as np
import matplotlib.pyplot as plt

hist = [json.loads(l) for l in open(RUN / "train_log.jsonl")]
w = 20
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for i, (k, lbl) in enumerate([("return", "episode return"), ("success", "success rate"), ("lifted", "lifted rate")]):
    v = np.array([h[k] for h in hist], float)
    s = np.convolve(v, np.ones(w) / w, "valid")
    ax[i].plot(np.arange(len(s)) + w, s)
    ax[i].set_xlabel("episode"); ax[i].set_title(f"{lbl} ({w}-episode mean)")
    if k != "return":
        ax[i].axhline(0.45, ls="--", c="grey", lw=1, label="base policy (45%)"); ax[i].legend()
plt.tight_layout(); plt.show()

for f in sorted(glob.glob(str(RUN / "eval_*.json"))):
    e = json.loads(Path(f).read_text()); p = e["paired"]
    print(f"{Path(f).stem:14} residual {e['residual']['success_rate']:5.0%}  "
          f"baseline {e['baseline']['success_rate']:5.0%}  (n={e['residual']['n']})  "
          f"paired +{p['residual_only_wins']} -{p['baseline_only_wins']}")

## Caveats

- **This base policy was trained on defective data.** The 2026-10-07 run learned from 33 episodes
  whose release detection fired mid-air; after the fix there are 51 usable episodes and a retrain is
  pending. Residual gains measured here may not survive a better base — which is precisely the
  question worth asking once the retrain lands.
- **`lifted` is a proxy for `grasped`.** It is cube-centre > 4 cm above resting, so a cube knocked
  into the air counts and collects the lift bonus. The drop penalty and the place term should
  punish it within a few steps, but it is the first reward-hacking surface to check.
- **The reach and place terms are distance penalties, not potentials**, so in principle they shift
  the optimum rather than only the learning dynamics. If hovering-near-the-cube behaviour appears,
  converting both to potential differences is the first fix.
- **Colab free gives ~2 vCPUs.** Once the GPU serves the policy, MuJoCo physics becomes the
  bottleneck, so extra parallel envs buy little without more cores.